<a href="https://colab.research.google.com/github/eigiihs/review-score-prediction/blob/giovanna/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto AWS: O Peso das Palavras
---

## Problema de Negócio
Em marketplaces digitais, nem sempre os clientes atribuem notas coerentes com seus comentários, ou até mesmo deixam de avaliar numericamente um produto. Dessa forma, surge a necessidade de desenvolver modelos que consigam interpretar automaticamente o sentimento do texto e convertê-lo em uma nota de satisfação.

## Objetivo
Construir um modelo preditivo capaz de estimar a nota de um produto (de 1 a 5) com base no conteúdo textual dos comentários.
O Desafio Técnico: utilizar um modelo de Regressão Linear para encontrar a relação entre o "tom" do texto e a "nota" atribuída.

In [ ]:
# !pip install pandas tqdm matplotlib seaborn nltk leia-br transformers torch pysentimiento

In [ ]:
# Gerais
import re
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import confusion_matrix
tqdm.pandas()
warnings.filterwarnings('ignore')

# Gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Análise de Sentimento
import nltk
from nltk.corpus import stopwords
from LeIA import SentimentIntensityAnalyzer as LEIAAnalyzer
from nltk.sentiment import SentimentIntensityAnalyzer as VaderAnalyzer
from transformers import pipeline
from pysentimiento import create_analyzer
from tqdm import tqdm
import torch

In [ ]:
nltk.download('stopwords')
nltk.download('vader_lexicon')

## 1. Leitura das tabelas

Nesta etapa carregamos a tabela do dataset que será utilizada no projeto.
O foco principal da análise está na tabela de **avaliações**, que contém informações textuais escritas pelos consumidores, além das notas atribuídas aos produtos.

A leitura inicial dos dados permite compreender:

- estrutura das tabelas
- quantidade de registros
- variáveis disponíveis
- valores nulos e duplicatas

In [ ]:
import pandas as pd
import os

path = './bronze'

dfs = {}

for file in os.listdir(path):
  if file.endswith('.csv'):
    df_name = "df_" + file.replace(".csv", "")
    full_path = os.path.join(path, file)

    dfs[df_name] = pd.read_csv(full_path)

In [ ]:
resultado = []

for nome, df in dfs.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    duplicatas = df.duplicated().sum()

    # Caso não tenha nulos
    if nulos.empty:
        resultado.append({
            "dataset": nome,
            "shape": df.shape,
            "coluna": None,
            "nulos": 0,
            "pct_nulos": 0.0,
            "duplicatas": duplicatas
        })
    else:
        for col, qtd in nulos.items():
            resultado.append({
                "dataset": nome,
                "shape": df.shape,
                "coluna": col,
                "nulos": qtd,
                "pct_nulos": round((qtd / len(df)) * 100, 1),
                "duplicatas": duplicatas
            })

df_resumo = pd.DataFrame(resultado)
display(df_resumo)

## 2. Transformação e Limpeza

### Tabela de Avaliações

Antes da aplicação dos modelos de análise de sentimento, os dados textuais passam por um processo de transformação e limpeza com o objetivo de comparar a qualidade das análises realizadas.

Nesta etapa serão realizadas:

- Validação dos tipos de dados
- Unificação dos campos textuais
- Tratamento de valores nulos
- Padronização da escrita
- Remoção de caracteres desnecessários
- Remoção de stopwords

### 2.1 Tipificação dos dados

In [ ]:
# STRINGS
dfs['df_avaliacoes']['review_id'] = dfs['df_avaliacoes']['review_id'].astype('string')
dfs['df_avaliacoes']['order_id'] = dfs['df_avaliacoes']['order_id'].astype('string')
dfs['df_avaliacoes']['review_comment_message'] = dfs['df_avaliacoes']['review_comment_message'].astype('string')
dfs['df_avaliacoes']['review_comment_title'] = dfs['df_avaliacoes']['review_comment_title'].astype('string')

# INT
dfs['df_avaliacoes']['review_score'] = dfs['df_avaliacoes']['review_score'].astype('int8') # int8 = economia de memória

# DATA
dfs['df_avaliacoes']['review_creation_date'] = pd.to_datetime(dfs['df_avaliacoes']['review_creation_date'], errors='coerce')
dfs['df_avaliacoes']['review_answer_timestamp'] = pd.to_datetime(dfs['df_avaliacoes']['review_answer_timestamp'], errors='coerce')

dfs['df_avaliacoes'].dtypes

### 2.2 Coluna de texto unificada e dados nulos

**2.2.1 Validando casos onde existe título e não existe comentário**

In [ ]:
total = len(dfs['df_avaliacoes'])
count = dfs['df_avaliacoes'][
    dfs['df_avaliacoes']["review_comment_title"].notna() & dfs['df_avaliacoes']["review_comment_message"].isna()
].shape[0]

percent = (count / total) * 100
print(f"{count} linhas ({percent:.2f}%)")

**2.2.2 Criando coluna unificada com título e comentário**

In [ ]:
df_avaliacao = dfs['df_avaliacoes'].copy()

# substitui nulos por string vazia para concatenação
df_avaliacao['review_comment_message'] = df_avaliacao['review_comment_message'].fillna('')
df_avaliacao['review_comment_title'] = df_avaliacao['review_comment_title'].fillna('')

df_avaliacao['review_text'] = (
    df_avaliacao['review_comment_title'] + ' ' + df_avaliacao['review_comment_message']
).str.strip()

# definindo valores vazios para "sem comentários"
df_avaliacao['review_text'] = df_avaliacao['review_text'].replace('', 'sem comentários')

percentual = round((df_avaliacao['review_text'] == 'sem comentários').mean() * 100,2)
print(f"Percentual de valores 'sem comentários' em 'review_text': {percentual}")
df_avaliacao[['review_comment_title', 'review_comment_message', 'review_text']].head(10)

**2.2.3 Removendo linhas sem comentários**

In [ ]:
df_avaliacao = df_avaliacao[
    df_avaliacao['review_text'] != 'sem comentários'
].copy()

print(f"Quantidade de linhas atuais: {df_avaliacao.shape[0]}")

### 2.3 Padronizando textos da coluna unificada "review_text"

**2.3.1 Padronização:**
- Tudo minúsculo
- Remoção de acentos e pontuações
- Remoção de espaços extras

In [ ]:
import unicodedata
import re

def standardize(text):
  if pd.isnull(text):
    return text

  # minúsculo
  text = text.lower()

  # remove acentos
  text = ''.join(
      c for c in unicodedata.normalize('NFD', text) # NFD: quebra caracteres acentuados em partes separadas (á -> "a" + "´")
      if unicodedata.category(c) != 'Mn' # valida o tipo do caractere (Mn: acentos e diacríticos)
  )

  # remover pontuações
  text = re.sub(r'[^\w\s]', '', text) # substitui tudo que não é letra\número\espaço por ""

  # remover múltiplos espaços
  text = re.sub(r'\s+', ' ', text).strip()

  # garante separação correta de palavras
  palavras = re.findall(r'\b\w+\b', text)
  text = ' '.join(palavras)

  return text

df_avaliacao['review_text_modificado'] = df_avaliacao['review_text'].apply(standardize)

**2.3.2 Removendo stopwords**

_Stopwords_ são palavras comuns que geralmente não carregam muito significado relevante para a análise, como: "o", "a", "de", "da", "para", "com"...

In [ ]:
stop_words = set(stopwords.words('portuguese'))

def remover_stopwords(texto):
    if pd.isna(texto):
        return None, []
    tokens = texto.split()
    removidas = [t for t in tokens if t in stop_words]
    tokens = [t for t in tokens if t not in stop_words]
    return (' '.join(tokens) if tokens else None), removidas

resultados = df_avaliacao['review_text_modificado'].apply(remover_stopwords)
df_avaliacao['review_text_modificado'] = resultados.apply(lambda x: x[0])
df_avaliacao['stopwords_removidas'] = resultados.apply(lambda x: x[1])

df_avaliacao = df_avaliacao[df_avaliacao['review_text_modificado'].notna()]

# Verificar exemplo
exemplo = df_avaliacao[df_avaliacao['review_text_modificado'].notna()].iloc[0]
removidas = exemplo['stopwords_removidas']

print(f"Nota: {exemplo['review_score']}")
print(f"Original:  {exemplo['review_text']}")
print(f"Modificado:     {exemplo['review_text_modificado']}")
print(f"\nQuantidade de stopwords removidas: {len(removidas)}")
print(f"Stopwords: {removidas}")

**2.3.3 Removendo colunas que não serão utilizadas**

In [ ]:
df_avaliacao = df_avaliacao.drop(columns=['order_id', 'review_creation_date','review_answer_timestamp','review_comment_title', 'review_comment_message', 'review_id'])

**2.3.4 Salvando base final**

In [ ]:
df_avaliacao.sample(5, random_state=32)

In [ ]:
df_avaliacao.to_parquet("./silver/df_avaliacao.parquet", index=False)

## 3. Análise de Sentimento

A análise de sentimento é uma técnica de Processamento de Linguagem Natural (NLP) utilizada para **identificar opiniões, emoções e polaridade presentes em textos**.

Neste projeto, o objetivo é verificar o quanto diferentes modelos conseguem representar corretamente a percepção do consumidor a partir dos comentários realizados nas avaliações.


In [ ]:
PARQUET_PATH = "./silver/df_avaliacao.parquet"
df_avaliacao = pd.read_parquet(PARQUET_PATH)

df_avaliacao.head(5)

**Função padrão que aplica o modelo de sentimento e retorna um DF padronizado**

In [ ]:
def aplicar_modelo_sentimento(df, nome_modelo, func_score, coluna_texto):
    """
    Parâmetros:
    - df: DataFrame original
    - nome_modelo: string com nome do modelo (ex: 'leia')
    - func_score: função que recebe texto e retorna score [-1, 1]
    - coluna_texto: coluna com o texto
    """

    df_result = df.copy()

    # Score
    df_result[f'{nome_modelo}_score'] = df_result[coluna_texto].progress_apply(func_score)

    # Retorno padronizado
    return df_result[[coluna_texto, 'review_score',
                      f'{nome_modelo}_score']]

**Classificação usada para modelos contínuos**

In [ ]:
def score_para_estrelas(score):

    if score <= -0.6:
        return 1

    elif -0.6 < score <= -0.2:
        return 2

    elif -0.2 < score <= 0.2:
        return 3

    elif 0.2 < score <= 0.6:
        return 4

    else:
        return 5

### 3.1 Vader

O **VADER (Valence Aware Dictionary and sEntiment Reasoner)** é um modelo de análise de sentimento baseado em:

- léxico de palavras
- regras linguísticas
- intensificadores e negações

Originalmente desenvolvido para textos em inglês, o VADER é amplamente utilizado por sua simplicidade e velocidade.

In [ ]:
vader = VaderAnalyzer()

def score_vader(text):
  return vader.polarity_scores(text)['compound']

df_vader = aplicar_modelo_sentimento(df_avaliacao, 'vader', score_vader, 'review_text')
df_vader_modificado = aplicar_modelo_sentimento(df_avaliacao, 'vader', score_vader, 'review_text_modificado')

comparacao_vader = pd.DataFrame({
    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    'vader_score_original': df_vader['vader_score'],
    'vader_score_modificado': df_vader_modificado['vader_score']
})

comparacao_vader.sample(10, random_state=32)

In [ ]:
comparacao_vader.to_parquet("./silver/df_vader.parquet")

In [ ]:
PATH = "./silver/df_vader.parquet"
vader = pd.read_parquet(PATH)

# Transformando score em estrelas
vader['vader_estrelas_original'] = (vader['vader_score_original'].apply(score_para_estrelas))
vader['vader_estrelas_modificado'] = (vader['vader_score_modificado'].apply(score_para_estrelas))

fig, axes = plt.subplots(1, 3, figsize=(20,5))

# GRÁFICO 1 - DISTRIBUIÇÃO REAL DAS NOTAS
sns.countplot(
    data=vader,
    x='nota_real',
    order=[1,2,3,4,5],
    palette='Set2',
    ax=axes[0]
)

axes[0].set_title('Distribuição Real das Notas')
axes[0].set_xlabel('Nota Real')
axes[0].set_ylabel('Quantidade')

# GRÁFICO 2 - VADER TEXTO ORIGINAL
sns.countplot(
    data=vader,
    x='vader_estrelas_original',
    order=[1,2,3,4,5],
    palette='Blues',
    ax=axes[1]
)

axes[1].set_title('VADER — Texto Original')
axes[1].set_xlabel('Estrelas Preditas')
axes[1].set_ylabel('Quantidade')

# GRÁFICO 3 - VADER TEXTO MODIFICADO
sns.countplot(
    data=vader,
    x='vader_estrelas_modificado',
    order=[1,2,3,4,5],
    palette='Greens',
    ax=axes[2]
)

axes[2].set_title('VADER — Texto Modificado')
axes[2].set_xlabel('Estrelas Preditas')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# CORRELAÇÕES
corr = pd.DataFrame({
    'Métrica': ['Pearson', 'Pearson', 'Spearman', 'Spearman'],
    'Texto': ['Original', 'Modificado', 'Original', 'Modificado'],
    'Correlação': [
        pearsonr(vader['nota_real'],vader['vader_estrelas_original'])[0],
        pearsonr(vader['nota_real'],vader['vader_estrelas_modificado'])[0],
        spearmanr(vader['nota_real'],vader['vader_estrelas_original'])[0],
        spearmanr(vader['nota_real'],vader['vader_estrelas_modificado'])[0]
    ]
})

# MATRIZ DE CONFUSÃO
cm_original = confusion_matrix(
    vader['nota_real'],
    vader['vader_estrelas_original'],
    labels=[1,2,3,4,5]
)

cm_modificado = confusion_matrix(
    vader['nota_real'],
    vader['vader_estrelas_modificado'],
    labels=[1,2,3,4,5]
)

# GRÁFICO
fig, axes = plt.subplots(1, 3, figsize=(18,5))

ax = sns.barplot(
    data=corr,
    x='Métrica',
    y='Correlação',
    hue='Texto',
    palette='viridis',
    ax=axes[0]
)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt='%.4f',
        padding=3,
        fontsize=8
    )

axes[0].set_title('Correlação — VADER')
axes[0].set_ylabel('Coeficiente')
axes[0].set_xlabel('')
axes[0].set_ylim(0,1)

axes[0].grid(axis='y', alpha=0.3)

# MATRIZES
sns.heatmap(
    cm_original,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[1]
)

axes[1].set_title('Matriz — Texto Original')
axes[1].set_xlabel('Predição')
axes[1].set_ylabel('Nota Real')

sns.heatmap(
    cm_modificado,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[2]
)

axes[2].set_title('Matriz — Texto Modificado')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Nota Real')

plt.tight_layout()
plt.show()

### 3.2 LeIA

O **LeIA** é uma adaptação do VADER para língua portuguesa.

Seu funcionamento é semelhante ao VADER, porém utilizando um léxico adaptado para português brasileiro, tornando-o mais adequado para comentários escritos por consumidores brasileiros.

In [ ]:
leia = LEIAAnalyzer()

def score_leia(text):
  return leia.polarity_scores(text)['compound']

df_leia = aplicar_modelo_sentimento(df_avaliacao, 'leia', score_leia, 'review_text')
df_leia_modificado = aplicar_modelo_sentimento(df_avaliacao, 'leia', score_leia, 'review_text_modificado')

comparacao_leia = pd.DataFrame({
    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    'leia_score_original': df_leia['leia_score'],
    'leia_score_modificado': df_leia_modificado['leia_score'],
})

comparacao_leia.sample(20, random_state=32)

In [ ]:
comparacao_leia.to_parquet("./silver/df_leia.parquet")

In [ ]:
PATH = "./silver/df_leia.parquet"
leia = pd.read_parquet(PATH)
leia.head(5)

In [ ]:
PATH = "./silver/df_leia.parquet"
leia = pd.read_parquet(PATH)

# Transformando score em estrelas
leia['leia_estrelas_original'] = (leia['leia_score_original'].apply(score_para_estrelas))
leia['leia_estrelas_modificado'] = (leia['leia_score_modificado'].apply(score_para_estrelas))

fig, axes = plt.subplots(1, 3, figsize=(16,5))

# GRÁFICO 1 - DISTRIBUIÇÃO REAL DAS NOTAS
sns.countplot(
    data=leia,
    x='nota_real',
    order=[1,2,3,4,5],
    palette='Set2',
    ax=axes[0]
)

axes[0].set_title('Distribuição Real das Notas')
axes[0].set_xlabel('Nota Real')
axes[0].set_ylabel('Quantidade')

# GRÁFICO 2 - LEIA TEXTO ORIGINAL
sns.countplot(
    data=leia,
    x='leia_estrelas_original',
    order=[1,2,3,4,5],
    palette='Blues',
    ax=axes[1]
)

axes[1].set_title('LEIA — Texto Original')
axes[1].set_xlabel('Estrelas Preditas')
axes[1].set_ylabel('Quantidade')

# GRÁFICO 3 - LEIA TEXTO MODIFICADO
sns.countplot(
    data=leia,
    x='leia_estrelas_modificado',
    order=[1,2,3,4,5],
    palette='Greens',
    ax=axes[2]
)

axes[2].set_title('LEIA — Texto Modificado')
axes[2].set_xlabel('Estrelas Preditas')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# CORRELAÇÕES
corr = pd.DataFrame({
    'Métrica': ['Pearson', 'Pearson', 'Spearman', 'Spearman'],
    'Texto': ['Original', 'Modificado', 'Original', 'Modificado'],
    'Correlação': [
        pearsonr(leia['nota_real'],leia['leia_estrelas_original'])[0],
        pearsonr(leia['nota_real'],leia['leia_estrelas_modificado'])[0],
        spearmanr(leia['nota_real'],leia['leia_estrelas_original'])[0],
        spearmanr(leia['nota_real'],leia['leia_estrelas_modificado'])[0]
    ]
})

# MATRIZ DE CONFUSÃO
cm_original = confusion_matrix(
    leia['nota_real'],
    leia['leia_estrelas_original'],
    labels=[1,2,3,4,5]
)

cm_modificado = confusion_matrix(
    leia['nota_real'],
    leia['leia_estrelas_modificado'],
    labels=[1,2,3,4,5]
)

# GRÁFICO
fig, axes = plt.subplots(1, 3, figsize=(18,5))

ax = sns.barplot(
    data=corr,
    x='Métrica',
    y='Correlação',
    hue='Texto',
    palette='viridis',
    ax=axes[0]
)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt='%.4f',
        padding=3,
        fontsize=8
    )

axes[0].set_title('Correlação — LEIA')
axes[0].set_ylabel('Coeficiente')
axes[0].set_xlabel('')
axes[0].set_ylim(0,1)

axes[0].grid(axis='y', alpha=0.3)

# MATRIZES
sns.heatmap(
    cm_original,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[1]
)

axes[1].set_title('Matriz — Texto Original')
axes[1].set_xlabel('Predição')
axes[1].set_ylabel('Nota Real')

sns.heatmap(
    cm_modificado,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[2]
)

axes[2].set_title('Matriz — Texto Modificado')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Nota Real')

plt.tight_layout()
plt.show()

### 3.3 Transforms
Os modelos **Transformer** representam o estado da arte em tarefas modernas de Processamento de Linguagem Natural.

Diferentemente dos modelos baseados apenas em palavras isoladas, Transformers conseguem compreender:

- contexto
- relações semânticas
- negação
- intensidade emocional
- dependências entre palavras

Por possuírem maior complexidade computacional, os testes serão realizados inicialmente em uma amostra da base de dados.

**3.3.1 TRANSFORMER 1 — XLM-RoBERTa**

O **XLM-RoBERTa (Robustly Optimized BERT Approach)** é um modelo Transformer multilíngue treinado em diversos idiomas, incluindo português.

O modelo é capaz de compreender contexto textual de forma muito mais profunda que modelos baseados apenas em léxico, permitindo melhor interpretação de:

- frases negativas
- sarcasmo leve
- negações
- contexto semântico

In [ ]:
# !pip install sentencepiece tiktoken protobuf

In [ ]:
# GPU / CPU
device = 0 if torch.cuda.is_available() else -1

print(f'\nUsando device: {device}')

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


# PIPELINE
classifier_xlmr = pipeline(
    task='sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    tokenizer='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    device=device
)


# CONVERSÃO SCORE -> ESTRELAS
def converter_para_estrelas(label, score):

    label = label.lower()

    if label == 'negative':

        if score >= 0.75:
            return 1
        else:
            return 2

    elif label == 'positive':

        if score >= 0.75:
            return 5
        else:
            return 4

    else:
        return 3

# EXECUTANDO INFERÊNCIA
print('\nExecutando inferência - texto original...')
textos_original = df_avaliacao['review_text'].tolist()

resultado_original = classifier_xlmr(
    textos_original,
    truncation=True,
    max_length=512,
    batch_size=32
)

print('\nExecutando inferência - texto modificado...')
textos_modificados = df_avaliacao['review_text_modificado'].tolist()
resultado_modificado = classifier_xlmr(
    textos_modificados,
    truncation=True,
    max_length=512,
    batch_size=32
)


# PROCESSAMENTO DOS RESULTADOS
def processar_resultados(resultados):

    linhas = []

    for r in resultados:

        try:

            label = r['label'].lower()
            score = round(r['score'], 4)

            estrelas = converter_para_estrelas(
                label,
                score
            )

            linhas.append({
                'label': label,
                'score': score,
                'estrelas': estrelas
            })

        except Exception as e:

            print(f'Erro ao processar resultado: {e}')

            linhas.append({
                'label': 'erro',
                'score': np.nan,
                'estrelas': np.nan
            })

    return pd.DataFrame(linhas)


# DATAFRAMES AUXILIARES
df_original = processar_resultados(resultado_original)
df_modificado = processar_resultados(resultado_modificado)


# COMPARAÇÃO FINAL
comparacao_roberta = pd.DataFrame({

    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    # ORIGINAL
    'label_original': df_original['label'],
    'score_original': df_original['score'],
    'estrelas_original': df_original['estrelas'],

    # MODIFICADO
    'label_modificado': df_modificado['label'],
    'score_modificado': df_modificado['score'],
    'estrelas_modificado': df_modificado['estrelas']

})

comparacao_roberta.sample(10, random_state=32)

In [ ]:
comparacao_roberta.to_parquet("df_roberta.parquet")

In [ ]:
PATH = "./silver/df_roberta.parquet"
roberta = pd.read_parquet(PATH)


fig, axes = plt.subplots(1, 3, figsize=(16,5))

# GRÁFICO 1 - DISTRIBUIÇÃO REAL DAS NOTAS
sns.countplot(
    data=roberta,
    x='nota_real',
    order=[1,2,3,4,5],
    palette='Set2',
    ax=axes[0]
)

axes[0].set_title('Distribuição Real das Notas')
axes[0].set_xlabel('Nota Real')
axes[0].set_ylabel('Quantidade')

# GRÁFICO 2 - XLM-RoBERTa TEXTO ORIGINAL
sns.countplot(
    data=roberta,
    x='estrelas_original',
    order=[1,2,3,4,5],
    palette='Blues',
    ax=axes[1]
)

axes[1].set_title('XLM-RoBERTa — Texto Original')
axes[1].set_xlabel('Estrelas Preditas')
axes[1].set_ylabel('Quantidade')

# GRÁFICO 3 - XLM-RoBERTa TEXTO MODIFICADO
sns.countplot(
    data=roberta,
    x='estrelas_modificado',
    order=[1,2,3,4,5],
    palette='Greens',
    ax=axes[2]
)

axes[2].set_title('XLM-RoBERTa — Texto Modificado')
axes[2].set_xlabel('Estrelas Preditas')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# CORRELAÇÕES
corr = pd.DataFrame({
    'Métrica': ['Pearson', 'Pearson', 'Spearman', 'Spearman'],
    'Texto': ['Original', 'Modificado', 'Original', 'Modificado'],
    'Correlação': [
        pearsonr(roberta['nota_real'],roberta['estrelas_original'])[0],
        pearsonr(roberta['nota_real'],roberta['estrelas_modificado'])[0],
        spearmanr(roberta['nota_real'],roberta['estrelas_original'])[0],
        spearmanr(roberta['nota_real'],roberta['estrelas_modificado'])[0]
    ]
})

# MATRIZ DE CONFUSÃO
cm_original = confusion_matrix(
    roberta['nota_real'],
    roberta['estrelas_original'],
    labels=[1,2,3,4,5]
)

cm_modificado = confusion_matrix(
    roberta['nota_real'],
    roberta['estrelas_modificado'],
    labels=[1,2,3,4,5]
)

# GRÁFICO
fig, axes = plt.subplots(1, 3, figsize=(18,5))

ax = sns.barplot(
    data=corr,
    x='Métrica',
    y='Correlação',
    hue='Texto',
    palette='viridis',
    ax=axes[0]
)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt='%.4f',
        padding=3,
        fontsize=8
    )

axes[0].set_title('Correlação — XLM-RoBERTa')
axes[0].set_ylabel('Coeficiente')
axes[0].set_xlabel('')
axes[0].set_ylim(0,1)

axes[0].grid(axis='y', alpha=0.3)

# MATRIZES
sns.heatmap(
    cm_original,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[1]
)

axes[1].set_title('Matriz — Texto Original')
axes[1].set_xlabel('Predição')
axes[1].set_ylabel('Nota Real')

sns.heatmap(
    cm_modificado,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[2]
)

axes[2].set_title('Matriz — Texto Modificado')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Nota Real')

plt.tight_layout()
plt.show()

**3.3.2 TRANSFORMER 1 — BERT Multilingual Sentiment**

Este modelo **BERT** foi treinado especificamente para prever avaliações em formato de estrelas (1 a 5), tornando sua aplicação especialmente interessante para problemas relacionados à previsão de notas de avaliações.

O modelo transforma o texto em representações vetoriais contextualizadas e estima o nível de satisfação do consumidor com base no conteúdo do comentário.

Por já possuir treinamento voltado para avaliações, espera-se uma forte relação entre seus resultados e as notas reais atribuídas pelos usuários.

In [ ]:
# GPU / CPU
device = 0 if torch.cuda.is_available() else -1
print(f'Usando device: {device}')

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


# PIPELINE
classifier_bert = pipeline(
    task='sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    tokenizer='nlptown/bert-base-multilingual-uncased-sentiment',
    device=device
)

# EXECUTANDO INFERÊNCIA
print('\nExecutando inferência - texto original...')
textos_original = df_avaliacao['review_text'].tolist()

resultado_original = classifier_bert(
    textos_original,
    truncation=True,
    max_length=512,
    batch_size=32
)

print('\nExecutando inferência - texto modificado...')
textos_modificados = df_avaliacao['review_text_modificado'].tolist()

resultado_modificado = classifier_bert(
    textos_modificados,
    truncation=True,
    max_length=512,
    batch_size=64
)

# FUNÇÃO SCORE
def processar_resultados(resultados):

    linhas = []

    for r in resultados:

        label = r['label']
        score = round(r['score'], 4)

        estrelas = int(label[0])

        linhas.append({
            'label': label,
            'score': score,
            'estrelas': estrelas
        })

    return pd.DataFrame(linhas)


# DATAFRAMES AUXILIARES
df_original = processar_resultados(resultado_original)
df_modificado = processar_resultados(resultado_modificado)


# COMPARAÇÃO FINAL
comparacao_bert = pd.DataFrame({

    'comentario': df_avaliacao['review_text'].values,
    'nota_real': df_avaliacao['review_score'].values,

    # ORIGINAL
    'score_original': df_original['score'].values,
    'estrelas_original': df_original['estrelas'].values,

    # MODIFICADO
    'score_modificado': df_modificado['score'].values,
    'estrelas_modificado': df_modificado['estrelas'].values
})

comparacao_bert.sample(10, random_state=32)

In [ ]:
comparacao_bert.to_parquet("df_bert.parquet")

In [ ]:
PATH = "./silver/df_bert.parquet"
bert = pd.read_parquet(PATH)


fig, axes = plt.subplots(1, 3, figsize=(16,5))

# GRÁFICO 1 - DISTRIBUIÇÃO REAL DAS NOTAS
sns.countplot(
    data=bert,
    x='nota_real',
    order=[1,2,3,4,5],
    palette='Set2',
    ax=axes[0]
)

axes[0].set_title('Distribuição Real das Notas')
axes[0].set_xlabel('Nota Real')
axes[0].set_ylabel('Quantidade')

# GRÁFICO 2 - BERT TEXTO ORIGINAL
sns.countplot(
    data=bert,
    x='estrelas_original',
    order=[1,2,3,4,5],
    palette='Blues',
    ax=axes[1]
)

axes[1].set_title('BERT — Texto Original')
axes[1].set_xlabel('Estrelas Preditas')
axes[1].set_ylabel('Quantidade')

# GRÁFICO 3 - BERT TEXTO MODIFICADO
sns.countplot(
    data=bert,
    x='estrelas_modificado',
    order=[1,2,3,4,5],
    palette='Greens',
    ax=axes[2]
)

axes[2].set_title('BERT — Texto Modificado')
axes[2].set_xlabel('Estrelas Preditas')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# CORRELAÇÕES
corr = pd.DataFrame({
    'Métrica': ['Pearson', 'Pearson', 'Spearman', 'Spearman'],
    'Texto': ['Original', 'Modificado', 'Original', 'Modificado'],
    'Correlação': [
        pearsonr(bert['nota_real'],bert['estrelas_original'])[0],
        pearsonr(bert['nota_real'],bert['estrelas_modificado'])[0],
        spearmanr(bert['nota_real'],bert['estrelas_original'])[0],
        spearmanr(bert['nota_real'],bert['estrelas_modificado'])[0]
    ]
})

# MATRIZ DE CONFUSÃO
cm_original = confusion_matrix(
    bert['nota_real'],
    bert['estrelas_original'],
    labels=[1,2,3,4,5]
)

cm_modificado = confusion_matrix(
    bert['nota_real'],
    bert['estrelas_modificado'],
    labels=[1,2,3,4,5]
)

# GRÁFICO
fig, axes = plt.subplots(1, 3, figsize=(18,5))

ax = sns.barplot(
    data=corr,
    x='Métrica',
    y='Correlação',
    hue='Texto',
    palette='viridis',
    ax=axes[0]
)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt='%.4f',
        padding=3,
        fontsize=8
    )

axes[0].set_title('Correlação — BERT')
axes[0].set_ylabel('Coeficiente')
axes[0].set_xlabel('')
axes[0].set_ylim(0,1)

axes[0].grid(axis='y', alpha=0.3)

# MATRIZES
sns.heatmap(
    cm_original,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[1]
)

axes[1].set_title('Matriz — Texto Original')
axes[1].set_xlabel('Predição')
axes[1].set_ylabel('Nota Real')

sns.heatmap(
    cm_modificado,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[2]
)

axes[2].set_title('Matriz — Texto Modificado')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Nota Real')

plt.tight_layout()
plt.show()

**3.3.3 PySentimiento**

O PySentimiento é uma biblioteca de análise de sentimento **baseada em modelos Transformer** treinados especificamente para tarefas de classificação textual em idiomas como português, espanhol, inglês e italiano.

O modelo utilizado para português foi treinado especificamente com textos em português, permitindo melhor adaptação à linguagem informal, abreviações e expressões comuns em redes sociais e avaliações online.

In [ ]:
# ANALYZER
analyzer = create_analyzer(
    task='sentiment',
    lang='pt'
)

# CONVERSÃO SCORE -> ESTRELAS
def converter_para_estrelas(label, score):

    label = label.upper()

    if label == 'NEG':

        if score >= 0.75:
            return 1
        else:
            return 2

    elif label == 'POS':

        if score >= 0.75:
            return 5
        else:
            return 4

    else:
        return 3

# EXECUTANDO INFERÊNCIA

print('\nExecutando inferência - texto original...')
textos_original = df_avaliacao['review_text'].tolist()

resultado_original = [
    analyzer.predict(texto)
    for texto in tqdm(textos_original)
]

print('\nExecutando inferência - texto modificado...')
textos_modificados = df_avaliacao['review_text_modificado'].tolist()

resultado_modificado = [
    analyzer.predict(texto)
    for texto in tqdm(textos_modificados)
]

# PROCESSANDO OS RESULTADOS
def processar_resultados(resultados):

    linhas = []

    for r in resultados:

        try:

            label = r.output.upper()

            score = round(
                r.probas[label],
                4
            )

            estrelas = converter_para_estrelas(
                label,
                score
            )

            linhas.append({

                'label': label,
                'score': score,
                'estrelas': estrelas
            })

        except Exception as e:

            print(f'Erro: {e}')

            linhas.append({

                'label': 'erro',
                'score': np.nan,
                'estrelas': np.nan
            })

    return pd.DataFrame(linhas)


# DATAFRAMES AUXILIARES
df_original = processar_resultados(resultado_original)
df_modificado = processar_resultados(resultado_modificado)

# COMPARAÇÃO FINAL
comparacao_pysentimiento = pd.DataFrame({

    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    # ORIGINAL
    'label_original': df_original['label'],
    'score_original': df_original['score'],
    'estrelas_original': df_original['estrelas'],

    # MODIFICADO
    'label_modificado': df_modificado['label'],
    'score_modificado': df_modificado['score'],
    'estrelas_modificado': df_modificado['estrelas']
})

comparacao_pysentimiento.sample(10, random_state=32)

In [ ]:
comparacao_pysentimiento.to_parquet("df_pysentimiento.parquet")


In [ ]:
PATH = "./silver/df_pysentimiento.parquet"
pysenti = pd.read_parquet(PATH)


fig, axes = plt.subplots(1, 3, figsize=(16,5))

# GRÁFICO 1 - DISTRIBUIÇÃO REAL DAS NOTAS
sns.countplot(
    data=pysenti,
    x='nota_real',
    order=[1,2,3,4,5],
    palette='Set2',
    ax=axes[0]
)

axes[0].set_title('Distribuição Real das Notas')
axes[0].set_xlabel('Nota Real')
axes[0].set_ylabel('Quantidade')

# GRÁFICO 2 - PYSENTIMIENTO TEXTO ORIGINAL
sns.countplot(
    data=pysenti,
    x='estrelas_original',
    order=[1,2,3,4,5],
    palette='Blues',
    ax=axes[1]
)

axes[1].set_title('PySentimiento — Texto Original')
axes[1].set_xlabel('Estrelas Preditas')
axes[1].set_ylabel('Quantidade')

# GRÁFICO 3 - PYSENTIMIENTO TEXTO MODIFICADO
sns.countplot(
    data=pysenti,
    x='estrelas_modificado',
    order=[1,2,3,4,5],
    palette='Greens',
    ax=axes[2]
)

axes[2].set_title('PySentimiento — Texto Modificado')
axes[2].set_xlabel('Estrelas Preditas')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# CORRELAÇÕES
corr = pd.DataFrame({
    'Métrica': ['Pearson', 'Pearson', 'Spearman', 'Spearman'],
    'Texto': ['Original', 'Modificado', 'Original', 'Modificado'],
    'Correlação': [
        pearsonr(pysenti['nota_real'],pysenti['estrelas_original'])[0],
        pearsonr(pysenti['nota_real'],pysenti['estrelas_modificado'])[0],
        spearmanr(pysenti['nota_real'],pysenti['estrelas_original'])[0],
        spearmanr(pysenti['nota_real'],pysenti['estrelas_modificado'])[0]
    ]
})

# MATRIZ DE CONFUSÃO
cm_original = confusion_matrix(
    pysenti['nota_real'],
    pysenti['estrelas_original'],
    labels=[1,2,3,4,5]
)

cm_modificado = confusion_matrix(
    pysenti['nota_real'],
    pysenti['estrelas_modificado'],
    labels=[1,2,3,4,5]
)

# GRÁFICO
fig, axes = plt.subplots(1, 3, figsize=(18,5))

ax = sns.barplot(
    data=corr,
    x='Métrica',
    y='Correlação',
    hue='Texto',
    palette='viridis',
    ax=axes[0]
)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt='%.4f',
        padding=3,
        fontsize=8
    )

axes[0].set_title('Correlação — PySentimiento')
axes[0].set_ylabel('Coeficiente')
axes[0].set_xlabel('')
axes[0].set_ylim(0,1)

axes[0].grid(axis='y', alpha=0.3)

# MATRIZES
sns.heatmap(
    cm_original,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[1]
)

axes[1].set_title('Matriz — Texto Original')
axes[1].set_xlabel('Predição')
axes[1].set_ylabel('Nota Real')

sns.heatmap(
    cm_modificado,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=[1,2,3,4,5],
    yticklabels=[1,2,3,4,5],
    ax=axes[2]
)

axes[2].set_title('Matriz — Texto Modificado')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Nota Real')

plt.tight_layout()
plt.show()

In [ ]:
vader = pd.read_parquet('df_vader.parquet')
leia = pd.read_parquet('df_leia.parquet')
roberta = pd.read_parquet('df_roberta.parquet')
bert = pd.read_parquet('df_bert.parquet')
pysenti = pd.read_parquet('df_senti')

from scipy.stats import pearsonr, spearmanr

correlacoes = pd.DataFrame({

    'Modelo': [
        'VADER', 'VADER',
        'LeIA', 'LeIA',
        'XLM-RoBERTa', 'XLM-RoBERTa',
        'BERT Multilingual', 'BERT Multilingual',
        'PySentimiento', 'PySentimiento'
    ],

    'Texto': [
        'Original', 'Modificado',
        'Original', 'Modificado',
        'Original', 'Modificado',
        'Original', 'Modificado',
        'Original', 'Modificado'
    ],

    'Pearson': [

        pearsonr(vader['nota_real'], vader['vader_estrelas_original'])[0],
        pearsonr(vader['nota_real'], vader['vader_estrelas_modificado'])[0],

        pearsonr(leia['nota_real'], leia['leia_estrelas_original'])[0],
        pearsonr(leia['nota_real'], leia['leia_estrelas_modificado'])[0],

        pearsonr(roberta['nota_real'], roberta['estrelas_original'])[0],
        pearsonr(roberta['nota_real'], roberta['estrelas_modificado'])[0],

        pearsonr(bert['nota_real'], bert['estrelas_original'])[0],
        pearsonr(bert['nota_real'], bert['estrelas_modificado'])[0],

        pearsonr(pysentimiento['nota_real'], pysentimiento['estrelas_original'])[0],
        pearsonr(pysentimiento['nota_real'], pysentimiento['estrelas_modificado'])[0],
    ],

    'Spearman': [

        spearmanr(vader['nota_real'], vader['vader_estrelas_original'])[0],
        spearmanr(vader['nota_real'], vader['vader_estrelas_modificado'])[0],

        spearmanr(leia['nota_real'], leia['leia_estrelas_original'])[0],
        spearmanr(leia['nota_real'], leia['leia_estrelas_modificado'])[0],

        spearmanr(roberta['nota_real'], roberta['estrelas_original'])[0],
        spearmanr(roberta['nota_real'], roberta['estrelas_modificado'])[0],

        spearmanr(bert['nota_real'], bert['estrelas_original'])[0],
        spearmanr(bert['nota_real'], bert['estrelas_modificado'])[0],

        spearmanr(pysentimiento['nota_real'], pysentimiento['estrelas_original'])[0],
        spearmanr(pysentimiento['nota_real'], pysentimiento['estrelas_modificado'])[0],
    ]
})

display(correlacoes.round(4))
